<a href="https://colab.research.google.com/github/AhmedMahmoud-123/FlyRank_AI/blob/main/work/notebooks/w03_feature_leakage_check.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-05 — Feature Vector and Leakage/Privacy Check

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [ ]:
import os

REPO_URL = 'https://github.com/AhmedMahmoud-123/FlyRank_AI.git'
REPO_DIR = '/content/FlyRank_AI'

if not os.path.exists(REPO_DIR):
    !git clone -q {REPO_URL}

os.chdir(REPO_DIR)
print('Working directory:', os.getcwd())

Working directory: /content/FlyRank_AI


In [ ]:
%pip -q install -r requirements.txt
!python scripts/01_prepare_features.py

Prepared 30,000 rows from 30,000 raw rows
Wrote /content/FlyRank_AI/FlyRank_AI/FlyRank_AI/data/processed/refresh_feature_vector.csv


In [ ]:
import sys
sys.path.append('scripts')

from ml_utils import MODEL_NUMERIC_FEATURES, MODEL_CATEGORICAL_FEATURES
print(MODEL_NUMERIC_FEATURES)

['search_volume', 'competition', 'cpc', 'word_count', 'char_count', 'log_impressions_90d', 'log_clicks_90d', 'log_sessions_90d', 'log_ai_sessions_90d', 'days_with_impressions', 'days_with_sessions', 'content_age_days', 'days_since_last_update', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct']


## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

In [ ]:
import pandas as pd

df = pd.read_csv('data/processed/refresh_feature_vector.csv')
print(f'{len(df):,} rows, {len(df.columns)} columns')

from scripts.ml_utils import MODEL_NUMERIC_FEATURES, MODEL_CATEGORICAL_FEATURES
print('Numeric features:', MODEL_NUMERIC_FEATURES)
print('Categorical features:', MODEL_CATEGORICAL_FEATURES)

30,000 rows, 52 columns
Numeric features: ['search_volume', 'competition', 'cpc', 'word_count', 'char_count', 'log_impressions_90d', 'log_clicks_90d', 'log_sessions_90d', 'log_ai_sessions_90d', 'days_with_impressions', 'days_with_sessions', 'content_age_days', 'days_since_last_update', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct']
Categorical features: ['competition_level', 'content_type', 'main_intent', 'age_tier', 'freshness_tier', 'word_count_tier', 'impression_tier', 'position_tier']


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

| Feature | Meaning | Available before label window? |
|---|---|---|
| `log_impressions_90d`, `log_clicks_90d`, `log_sessions_90d`, `log_ai_sessions_90d` | log-scaled 90-day totals | **No — see Section 3.** A 90-day window ending "now" necessarily contains the last-30-day window the label is derived from. |
| `content_age_days`, `days_since_last_update` | age / staleness | Yes — knowable before prediction |
| `avg_position`, `ctr`, `engagement_rate`, `scroll_rate` | prev+current performance stats | **Ambiguous — requires verification of the source window before use in the final model.**
| `word_count`, `char_count`, `search_volume`, `competition`, `cpc` | static content/keyword metadata | Yes — fixed at publish time |
| `competition_level`, `content_type`, `main_intent`, `age_tier`, `freshness_tier`, `word_count_tier`, `impression_tier`, `position_tier` | categorical bins | Mostly yes, except `impression_tier`/`position_tier` if binned from the same 90d-blended numbers |

## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

In [ ]:
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

y = df['is_declining_label']
X_all = df[MODEL_NUMERIC_FEATURES].fillna(0)

# 1. Correlation scan
print(X_all.corrwith(y).sort_values(key=abs, ascending=False))

# 2. Window-overlap logic check: is_declining_label comes from trend_direction, which compares
# impressions_last_30d vs impressions_prev_30d. The *_90d columns are built from raw impressions_90d --
# does impressions_90d = last_30d + prev_30d + earlier? If so, the 90d features structurally
# contain the label window.
sample = df[['impressions_90d', 'impressions_last_30d', 'impressions_prev_30d']].head(10)
print(sample)
print("does impressions_90d >= last_30d + prev_30d for most rows?",
      (df['impressions_90d'] >= df['impressions_last_30d'] + df['impressions_prev_30d']).mean())

# 3. Train WITH vs WITHOUT the suspect 90d-blended features
suspects = ['log_impressions_90d', 'log_clicks_90d', 'log_sessions_90d', 'log_ai_sessions_90d']
X_without = X_all.drop(columns=suspects)

score_with = cross_val_score(RandomForestClassifier(n_estimators=150, random_state=42), X_all, y, cv=5).mean()
score_without = cross_val_score(RandomForestClassifier(n_estimators=150, random_state=42), X_without, y, cv=5).mean()
print(f'CV accuracy WITH 90d-blended features:    {score_with:.3f}')
print(f'CV accuracy WITHOUT 90d-blended features: {score_without:.3f}')
print(f'base rate (majority class): {max(y.mean(), 1-y.mean()):.3f}')

# 4. Verification the test harness itself works: deliberately inject a copy of the label as a feature
X_leaky = X_without.copy()
X_leaky['fake_leak'] = y  # obviously circular, should send accuracy toward 1.0
leak_score = cross_val_score(RandomForestClassifier(n_estimators=150, random_state=42), X_leaky, y, cv=5).mean()
print(f'sanity check -- CV accuracy WITH an injected leaky copy of the label: {leak_score:.3f} (should be ~1.0)')

# 5. Product-flag check
product_flags = {'competition_level'}

print(
    "product/decision flags in configured model features:",
    set(MODEL_NUMERIC_FEATURES + MODEL_CATEGORICAL_FEATURES) & product_flags
)

print(
    "product/decision flags actually used by this model:",
    set(X_all.columns) & product_flags
)

days_with_impressions     0.190055
log_impressions_90d       0.177473
content_age_days         -0.163882
word_count                0.118863
char_count                0.108025
days_since_last_update    0.081383
ctr                      -0.061911
avg_position             -0.029035
days_with_sessions       -0.025055
log_sessions_90d          0.015270
search_volume            -0.013817
engagement_rate          -0.012743
competition               0.012575
cpc                      -0.006031
log_ai_sessions_90d      -0.004304
log_clicks_90d            0.003469
scroll_rate              -0.002711
ai_traffic_pct            0.002435
dtype: float64
   impressions_90d  impressions_last_30d  impressions_prev_30d
0             3803                   578                   987
1            15320                  2501                  5915
2            12581                  2382                  6089
3            11751                  3626                  4206
4            19140                  4211

**Leakage hunt result:** `impressions_90d` is structurally larger than `impressions_last_30d + impressions_prev_30d` for every row checked, which confirms that the 90-day window contains the two 30-day windows. Because the label is based on the last-30-day versus previous-30-day comparison, the 90-day features overlap the label window and should not be used in the final forward-looking model.

However, the model comparison does not show a large performance drop: cross-validation accuracy changes from 0.703 with the 90-day features to 0.701 without them. I therefore treat the exclusion as a temporal-validity decision, not as evidence that the 90-day features were responsible for the model's performance.

The injected-label test reaches 1.000 accuracy, confirming that the test setup can detect an obviously circular feature.

## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

| Excluded field | Why |
|---|---|
| `log_impressions_90d`, `log_clicks_90d`, `log_sessions_90d`, `log_ai_sessions_90d` | 90-day window structurally overlaps the last-30-day label window — confirmed leakage in Section 3 |
| `impressions_last_30d`, `clicks_last_30d`, `sessions_last_30d` | Directly the label window itself |
| `trend_pct`, `trend_direction` | The label is derived from these — using them is circular |
| `competition_level` | Excluded from the final model feature set because the project framing treats it as descriptive metadata rather than a decision feature. |
| `impression_tier`, `position_tier` | Excluded because their underlying performance measures may include the overlapping 90-day window; they are not used until their source window is verified. |
| `client_id`, `content_id` | Grouping/join keys only — never model inputs |

**What's left as legitimate features:** `content_age_days`, `days_since_last_update`, `word_count`, `char_count`, `search_volume`, `competition`, `cpc`, and categorical metadata whose definitions are known to be available before the prediction window. `avg_position`, `ctr`, `engagement_rate`, and `scroll_rate` remain subject to window verification.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.